![](https://site.unibo.it/rawmatcop-alliance/en/@@images/aa4c85c2-ecf6-48e4-a412-d3b49dc48ae7.png)

<style>
body {
  font-family: Arial, sans-serif;
  line-height: 1.6;
}

h1, h2 {
  color: #333;
}

h1 {
  border-bottom: 1px solid #3B8A4C;
  padding-bottom: 10px;
  margin-bottom: 20px;
}

h2 {
  margin-top: 40px;
}

code {
  font-family: Consolas, monospace;
  font-size: 14px;
  background-color: #f9f9f9;
  border: 1px solid #ccc;
  padding: 5px;
}

.graph-image {
  max-width: 100%;
  height: auto;
  margin-bottom: 20px;
}

</style>
<a name="Title">
  <hr color="#3B8A4C">
  <font size="7" color="#3B8A4C">Mapping the extent of mountaintop mining </br>in the Appalachian Mountains</br>for the period 1985-2023</font>
  <hr color="#3B8A4C">
</a>


#<a name="Section 1"><hr color="#3B8A4C"><font size="6" color="#3B8A4C">**1. Background**</font><hr color="#3B8A4C"></a>

#### <a name="Section 1.1"><font size="5" color="#1F497D">**1.1 Overview** </font></a>

Mountaintop mining (MTM) consists in the extraction of coal seams from a mountain by removing the land, or overburden, above the seams. To do so explosives are used to remove up to 400 vertical feet (120 m) of mountain and the excess rocks and soils are dumped into nearby valleys.

This method has been extensively conducted in the Appalachian Mountains (eastern United States) since the 60s. MTM occurs most commonly in West Virginia and Eastern Kentucky. In all an estimated 5,900 km2 of land have been affected.

This process present economic benefits and is considered to be safer compared to underground mining because the coal seams are accessed from above instead of underground. On the other hand, the practice of MTM has been controversial due to the negative impact not only on the environment and landscapes (destruction of mountains, burial of streams, deforestation and release of pollutants), but also on the health of the local populations due to air and water contamination.

#### <a name="Section 1.2"><font size="5" color="#1F497D">**1.2 Study area** </font></a>

The study area is located west of the town of Beckley (West Virginia).

This area has been severely impacted by mountaintop removal since the 80s.

In [ ]:
 #@title Location of the site
%pip install folium -q
import folium
from IPython.display import clear_output
clear_output()

# Set the latitude and longitude coordinates
latitude  =  37.900  # Example latitude
longitude = -81.700  # Example longitude

# Create a map centered at the specified latitude and longitude
map = folium.Map(location=[latitude, longitude], zoom_start=6, width ='66%')

# Add a marker at the specified latitude and longitude
folium.Marker(location=[latitude, longitude]).add_to(map)

# Display the map
map

#### <a name="Section 1.3"><font size="5" color="#1F497D">**1.3 Data** </font></a>

This exercise will mainly be based on Landsat optical imagery covering the period 1985-2023. Because of the extreme cloud cover in this area cloudless composite scenes were created for periods of 3 years by using a median filter:
* LANDSAT_composite_1985_to_1987.tif
* LANDSAT_composite_1988_to_1990.tif
* LANDSAT_composite_1991_to_1993.tif
* LANDSAT_composite_1994_to_1996.tif
* LANDSAT_composite_1997_to_1999.tif
* LANDSAT_composite_2000_to_2002.tif
* LANDSAT_composite_2003_to_2005.tif
* LANDSAT_composite_2006_to_2008.tif
* LANDSAT_composite_2009_to_2011.tif
* LANDSAT_composite_2012_to_2014.tif
* LANDSAT_composite_2015_to_2017.tif
* LANDSAT_composite_2018_to_2020.tif
* LANDSAT_composite_2021_to_2023.tif'

The scenes belongs to 3 generations of Landsat (5 TM, 7 ETM+, 8 OLI/TIRS). In order to simplify the processing for this exercise only the bands with a compatible wavelength between these 3 sensors are used:
* B1: 0.45-0.515 μm (Blue)
* B2: 0.52-0.605 μm (Green)
* B3: 0.63-0.69 μm  (Red)
* B4: 0.75-0.90 μm  (NIR)
* B5: 1.55-1.75 μm  (shortwave infrared 1)
* B6: 2.08-2.35 μm (shortwave infrared 2)

Characteristic spectra for bare soils/rock, vegetation, and water as provided as a csv file.

The input data can be downloaded from: [OneDrive](https://aghedupl-my.sharepoint.com/:f:/g/personal/mlupa_agh_edu_pl/IgC_uhBJqGh8SqUWoQdgVfllAcMQR1LVji1WpYpFBD8ZV64?e=k4CJUi)
Place the contents in the `data/` folder next to this notebook.


#<a name="Section 2"><hr color="#3B8A4C"><font size="6" color="#3B8A4C">**2. Notebook setup**</font><hr color="#3B8A4C"></a>

We install rasterio 1.4:

In [ ]:
%pip install -q rasterio>=1.3 scikit-image scikit-learn opencv-python graphviz pydotplus folium


We now import the packages that will be needed.

In [ ]:
import sys, os

# On Windows, PostgreSQL/PostGIS ships an outdated PROJ database that
# conflicts with rasterio. This forces rasterio to use its own copy.
if sys.platform == 'win32':
    try:
        import pyproj
        os.environ.setdefault('PROJ_DATA', pyproj.datadir.get_data_dir())
    except ImportError:
        pass

import os, sys
import copy
import numpy as np
import pandas
from datetime import datetime
import rasterio
from rasterio.plot import show as rasterio_show
import graphviz
import pydotplus
from PIL import Image
import cv2
from matplotlib import pyplot
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.tree import export_graphviz
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV
from skimage.measure import label

import warnings
warnings.filterwarnings('ignore')

We define a function to print a progress bar.

In [ ]:
def _clock():
    time = datetime.strftime(datetime.now(), '%H:%M:%S')
    return time

def print_progress_bar(index, total, label):
    n_bar = 50  # Progress bar width
    progress = index / total
    sys.stdout.write('\r')
    sys.stdout.write(_clock() + '   ' + f"[{'=' * int(n_bar * progress):{n_bar}s}] {int(100 * progress)}%  {label}")
    sys.stdout.flush()

#<a name="Section 3"><hr color="#3B8A4C"><font size="6" color="#3B8A4C">**3. Import and explore input data**</font><hr color="#3B8A4C"></a>

The Landsat composites and spectral training samples are already in the `data/` folder. The cell below just verifies everything is in place and shows a quick summary of the available scenes.

In [ ]:
import os
from pathlib import Path

scenes_dir = Path('data/landsat_scenes')
csv_path   = Path('data/spectral_samples.csv')

scenes = sorted(scenes_dir.glob('*.tif')) if scenes_dir.exists() else []
print(f'{len(scenes)} Landsat composite(s) found:')
for p in scenes:
    print(' ', p.name)
print()
print('spectral_samples.csv:', 'OK' if csv_path.exists() else 'MISSING')


We now have a look at one of the landsat scenes located in the /landsat_scenes folder:

In [ ]:
# input the path to the scene corresponding to July 1st, 2018
path = 'data/landsat_scenes/LANDSAT_composite_2021_to_2023.tif'
raster = rasterio.open(path, 'r')

# get the array dimensions
cols, rows, bands = raster.width, raster.height, raster.count
# get coordinate reference system
crs = raster.crs
# get the transform
geotransform = raster.transform

# print the information
print("Columns \n", cols)
print("Rows \n", rows)
print("Bands \n", bands)
print("CRS \n", crs)
print("Transform")
print(geotransform)

# plot the scene
fig = pyplot.figure(figsize=(15,15))
vmax = np.nanpercentile(raster.read([3,2,1]), 99.5)
normRGB = raster.read([3,2,1])/vmax
name = os.path.split(raster.name)[-1]
rasterio.plot.show(normRGB, transform=raster.transform, title='Natural Color visualisation of '+name);

We import the spectral samples (csv file):

In [ ]:
labels = []
values = []

data = pandas.read_csv("data/spectral_samples.csv")
labels = data[data.columns[0]].tolist()
values = data[data.columns[1::]].values

reference_spectra = {}
for key in np.unique(labels):
    ind = np.where(np.array(labels)==key)[0]
    reference_spectra[key] = values[ind,:]

print('\n')
for key in reference_spectra.keys():
    print(key + ' contains ' + str(reference_spectra[key].shape[0]) + ' spectras')

We visualize the spectral samples:

In [ ]:
# Landsat bands wavelength (nm)
wavelength = [np.mean((450,515)), np.mean((525,605)), np.mean((630,690)),
              np.mean((750,900)), np.mean((1550,1750)), np.mean((2080,2350))]

spectras = ['Water', 'Soil', 'Vegetation']
colors   = ['#001DD1', '#6D5026', '#2A5E2E']

fig = pyplot.figure(figsize=(24,6))
n=1
for spectrum, color in zip(spectras, colors):
    ax = fig.add_subplot(int('13'+str(n)))
    q5, q95 = np.nanpercentile(reference_spectra[spectrum], [5, 95], axis = 0)
    x = np.hstack((q5,q95[::-1]))
    y = np.hstack((wavelength,wavelength[::-1]))
    polygon = Polygon(np.vstack([y,x]).T)
    p = PatchCollection([polygon], color=color, alpha=0.3)
    ax.add_collection(p)
    ax.plot(wavelength, np.nanmean(reference_spectra[spectrum], axis = 0), 'o-', color=color);
    ax.set_ylim([0, 4000])
    ax.set_title(spectrum, fontsize=18);
    n+=1
ax.set_ylim([0, 4000])
pyplot.tight_layout();

#<a name="Section 5"><hr color="#3B8A4C"><font size="6" color="#3B8A4C">**4. Spectral indices**</font><hr color="#3B8A4C"></a>

We can test the NDVI to enhance vegetation features. In our case the NIR band is 4 and the red band is 3.

In [ ]:
# input the path to the scene corresponding to July 1st, 2018
path = 'data/landsat_scenes/LANDSAT_composite_2021_to_2023.tif'

# import the geotiff using rasterio
raster = rasterio.open(path, 'r')

# calculate the NDVI (in our dataset the NIR band is 4 and the red band is 3)
NDVI = (raster.read([4])-raster.read([3]))/(raster.read([4])+raster.read([3]))

vmin = np.nanpercentile(NDVI,5)
vmax = np.nanpercentile(NDVI,95)
# plot the  NDVI
fig = pyplot.figure(figsize=(10,10))
ax1 = fig.add_subplot(111)
ax1.set_title('NDVI')
map = ax1.imshow(NDVI[0,...], cmap='jet', vmin=vmin, vmax=vmax)
divider = make_axes_locatable(ax1)
cax = divider.append_axes("right", size="2%", pad=0.05)
cbar = fig.colorbar(map, cax=cax)
fig.tight_layout();

We can use spectral angles to enhance pixels with soil:

In [ ]:
# Create a reference spectrum by computing the median of all spectra from the 'AMD'
# array in the reference_spectra dictionary
ref_spectrum = np.nanmedian(reference_spectra['Soil'], axis = 0)

# Get the spectra from the imported scene
# first, import all the bands
landsatSpectra = raster.read()

# get the array dimensions
cols, rows, bands = raster.width, raster.height, raster.count
# then,  reshape the array in order to have for each row one spectrum
landsatSpectra = landsatSpectra.reshape((bands, rows * cols))

# our reshape array has 12 rows and n columns, we want the opposite
# we transpose the array
landsatSpectra = landsatSpectra.T

# now we compute the spectral angles between the spectra from the scene and our reference spectrum
dot_product = np.dot(landsatSpectra, ref_spectrum)
norm_product = np.linalg.norm(landsatSpectra, axis=1) * np.linalg.norm(ref_spectrum)
sam = np.arccos(np.divide(dot_product, norm_product))
sam = sam.reshape((rows, cols))

# now we can plot our map
vmin = 0
vmax = np.nanpercentile(sam,95)
fig = pyplot.figure(figsize=(8,8))
ax1 = fig.add_subplot(111)
ax1.set_title('Spectral angles with respect to Soil reference spectrum')
map = ax1.imshow(sam, cmap='jet_r', vmin=vmin, vmax=vmax)
divider = make_axes_locatable(ax1)
cax = divider.append_axes("right", size="2%", pad=0.05)
cbar = fig.colorbar(map, cax=cax)
fig.tight_layout();

#<a name="Section 7"><hr color="#3B8A4C"><font size="6" color="#3B8A4C">**5. Landcover classification**</font><hr color="#3B8A4C"></a>

## <a name="Subsection 6.1"><font size="5" color="#1F497D">**5.1 Creation of the training and validation datasets**</font></a>

In [ ]:
# create empty variables for the samples array (X) and the labels list (y)
X, y = None, None

# iterate throught the reference_spectra dictionary
for key in reference_spectra.keys():
    n_samples = reference_spectra[key].shape[0]
    print('The class', key, 'contains', n_samples, "samples.")
    if X is None and y is None:
        X = reference_spectra[key]
        y = [key] * reference_spectra[key].shape[0]
    else:
        y = y + ([key] * reference_spectra[key].shape[0])
        X = np.vstack((X, reference_spectra[key]))


print('\nThe samples array has', X.shape[0], ' rows and', X.shape[1], "columns:")
print(X)
print('\nThe labels list has', len(y), "elements:")
print(y[0:3], '...', y[-3::])

print('\nAppending spectral indices...')
# create a list for the spectral angles
sam_list = []

# for each class compute the angles and add the output array to the list
for item in ['Soil', 'Water']:
    print('- sam', item)
    ref_spectra = np.nanmedian(reference_spectra[item], axis = 0)
    sam = np.arccos(np.divide(np.dot(X, ref_spectra), (np.linalg.norm(X, axis=1) * np.linalg.norm(ref_spectra))))
    sam = sam.reshape((sam.shape[0],1))
    sam_list.append(sam)

# stack each array in the list to the training samples
for sam in sam_list:
    X = np.hstack((X, sam))

# compute NDVI (be careful of the indices in the np array, index of band 4 is 3 and index of band 3 is 2!)
print('- NDVI', item)
NDVI = (X[:,3]-X[:,2])/(X[:,3]+X[:,2])
NDVI = NDVI.reshape((NDVI.shape[0],1))
# stack NDVI values with the training samples
X = np.hstack((X, NDVI))

np.set_printoptions(precision=3)
np.set_printoptions(suppress=True)
print('\nThe samples array has', X.shape[0], ' rows and', X.shape[1], "columns:")
print(X)

# split samples (X) and labels (y)
# we use 25% for the test dataset (0.25)
# we pass y to the stratify argument (i.e. labels will be used)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y)

# Get the number of values for each labels and print them
values_train, counts_train = np.unique(y_train, return_counts=True)
print("\nComposition of the training dataset:")
for val, cnt in zip(values_train, counts_train):
    print(val, (10-len(val))*" ", cnt)
values_test, counts_test = np.unique(y_test, return_counts=True)
print("\nComposition of the validation dataset:")
for val, cnt in zip(values_test, counts_test):
    print(val, (10-len(val))*" ", cnt)

## <a name="Subsection 7.2"><font size="5" color="#1F497D">**5.2 RandomForest Classifier**</font></a>

Finding best parameters using scikit-learn's GridSearchCV:

In [ ]:
# we set a dictionary with the parameters we want to input and the values we want to test
# we set 'class_weight' to 'balanced' as the number of samples for each class is uneven,
# this adjust weights inversely proportional to class frequencies
parameters = [{ 'n_estimators'      : [50, 100, 200],        # number of trees
                'max_features'      : ["sqrt", "log2", None], # number of features to consider when looking for the best split
                'max_depth'         : [7, 9, 12],             # maximum depth of the tree
                'max_leaf_nodes'    : [None],                 # maximum number of leaves in a tree
                'class_weight'      : ['balanced'],           # The “balanced” mode automatically adjust weights inversely proportional to class frequencies
                'max_samples'       : [0.33],                 # the number of samples to draw from X to train each base estimator
                'min_samples_leaf'  : [10],                   # The minimum number of samples required to be at a leaf node.
                'n_jobs'            : [-1]
              },]

# We set the GridSearchCV. The accuracy will be used to rank the models
# the cv parameters sets the number of folds for the cross validation
clf = GridSearchCV(estimator=RandomForestClassifier(),
                   param_grid = parameters,
                   cv=3,
                   scoring="accuracy",
                   verbose=2,
                   )

# we run the grid search.
clf.fit(X_train, y_train)
print("\nBest parameters:", clf.best_params_)

We can have a look at the scores from the different models:

In [ ]:
print('mean scores:')
print(clf.cv_results_['mean_test_score'])
print('\nranking:')
print(clf.cv_results_['rank_test_score'])

We now create our classifier using the best parameters:

In [ ]:
# create a RandomForestClassifier with the best parameters
clf = RandomForestClassifier(n_estimators     = clf.best_params_['n_estimators'],
                             max_features     = clf.best_params_['max_features'],
                             max_depth        = clf.best_params_['max_depth'],
                             class_weight     = clf.best_params_['class_weight'],
                             max_samples      = clf.best_params_['max_samples'],
                             min_samples_leaf = clf.best_params_['min_samples_leaf'],
                             max_leaf_nodes   = clf.best_params_['max_leaf_nodes'],
                             n_jobs           = -1)

# Fit the classifier using the training data
clf.fit(X_train, y_train)

In [ ]:
ind=-1  #@param { type: "integer" }
max_depth=3  #@param { type: "integer" }

if ind==-1:
    ind = np.random.randint(0, clf.n_estimators, size=1)[0]

try:
    print('Plotting decision tree #', ind)
    # Export the first three decision trees from the forest
    feature_names = ["B1","B2","B3","B4","B5","B6", 'sam_Soil', 'sam_Water', "NDVI"]

    tree = clf.estimators_[ind]
    dot_data = export_graphviz(tree,
                                feature_names=feature_names,
                                filled=True,
                                max_depth=max_depth,
                                impurity=True,
                                proportion=True)
    graph = graphviz.Source(dot_data)

    #display(graph)

    # Here we can edit the nodes in order to color only the final nodes
    # and set the colors according to the class
    # the result is exported as a png file in the /content folder

    graph = pydotplus.graph_from_dot_data(dot_data)
    nodes = graph.get_node_list()
    # colors for each class sorted in alphabetic order
    # ['Soil', 'Vegetation', 'Water', ]
    colors = ["#C19D8E", "#48804F", "#344AD8"]
    for node in nodes:
        if node.get_name() not in ('node', 'edge', '"\\n"'):
            values = clf.estimators_[ind].tree_.value[int(node.get_name())][0]
            #color only nodes where only one class is present
            if max(values) == sum(values):
                node.set_fillcolor(colors[np.argmax(values)])
            #mixed nodes get the default color
            else:
                node.set_fillcolor('#FFFFFF')
    graph.write_png('data/tree.png')
    image = Image.open('data/tree.png').convert("RGB")
    opencvImage = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    fig_tree, ax_tree = pyplot.subplots(figsize=(16, 6))
    ax_tree.imshow(cv2.cvtColor(opencvImage, cv2.COLOR_BGR2RGB))
    ax_tree.axis('off')
    pyplot.tight_layout()
    pyplot.show()
except:
    print('Could not plot the tree. Check if the index is valid (ind <= n_trees)')

Look at feature importance based on decrease in impurity:

In [ ]:
feature_names = ["B1","B2","B3","B4","B5","B6", 'sam_Soil', 'sam_Water', "NDVI"]
importances = clf.feature_importances_
std = np.std([tree.feature_importances_ for tree in clf.estimators_], axis=0)
forest_importances = pandas.Series(importances, index=feature_names)
forest_importances = pandas.Series.sort_values(forest_importances, axis=0, ascending=False)
fig, ax = pyplot.subplots()
forest_importances.plot.bar(yerr=None, ax=ax)
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()

## <a name="Subsection 7.3"><font size="5" color="#1F497D">**5.3 Evaluating the classifier**</font></a>

In [ ]:
# Make predictions for the test set
y_pred = clf.predict(X_test)

# View accuracy score
print("Classifier accuracy (fraction of correctly classified samples):")
acc = accuracy_score(y_test, y_pred)
print("{:.2f}".format(acc*100), "%")

values_test, counts_test = np.unique(y_test, return_counts=True)
print("\nComposition of the test dataset:")
for val, cnt in zip(values_test, counts_test):
    print(val, (10-len(val))*" ", cnt)

print("\n")

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
# print a good looking table using pandas dataframe
text = ["Soil", "Vegetation", "Water"]
df = pandas.DataFrame(cm, columns=text)
df.insert(loc=0, column=' ', value=text)
df = df.style.set_table_styles([dict(selector='th', props=[('text-align', 'center')])])
df.set_properties(**{'text-align': 'center'}).hide()
display(df)

## <a name="Subsection 7.4"><font size="5" color="#1F497D">**5.4 Testing the classifier on a scene**</font></a>

Import a raster and make a 2D array as input for the classifier

In [ ]:
path = 'data/landsat_scenes/LANDSAT_composite_2018_to_2020.tif'

# open the scene using rasterio
raster = rasterio.open(path, 'r')
# get the array
scene = raster.read()

# get dimensions of the initial array
cols, rows, bands = raster.width, raster.height, raster.count
# Now we reshape the array from 3D (bands, rows, cols) to 2D (bands, pixels)
spectra = scene.reshape(bands, cols*rows)
# We transpose the array to have the (pixels, bands)
spectra = spectra.T

# compute the spectral angles
sam_list = []
for item in ['Soil', 'Water']:
    ref_spectra = np.nanmedian(reference_spectra[item], axis = 0)
    sam = np.arccos(np.divide(np.dot(spectra, ref_spectra), (np.linalg.norm(spectra, axis=1) * np.linalg.norm(ref_spectra))))
    sam = sam.reshape((sam.shape[0],1))
    sam_list.append(sam)
# stack the spectral angles to the array
for sam in sam_list:
    spectra = np.hstack((spectra, sam))

# compute NDVI (be careful of the indices in the np array, index of band 4 is 3 and index of band 3 is 2!)
NDVI = (spectra[:,3]-spectra[:,2])/(spectra[:,3]+spectra[:,2])
NDVI = NDVI.reshape((NDVI.shape[0],1))
# stack NDVI values with the training samples
spectra = np.hstack((spectra, NDVI))

print('Shape of the transposed array:', spectra.shape, '\n')
print(spectra)

Make predictions:

In [ ]:
y_pred = clf.predict(spectra)

# Get the number of values for each labels and print them
values_train, counts_train = np.unique(y_pred, return_counts=True)
print("Composition of the predicted dataset:")
for val, cnt in zip(values_train, counts_train):
    print(val, (10-len(val))*" ", cnt)

Plot the map

In [ ]:
# Create a dictionnary to translate from text to number
labels_as_numbers = {'Water':1, 'Soil':2, 'Vegetation':3}
# Iterate through the prediction list to 'translate' and convert the list to a numpy array
y_num = np.array([labels_as_numbers[i] for i in y_pred])
# Reshape the list in order to get a 2D array
y_num = y_num.reshape(rows, cols)

# we make a normalized RGB image
vmax = np.nanpercentile(raster.read([3,2,1]), 99.5)
normRGB = raster.read([3,2,1])/vmax

# create a colormap using 3 colors for our 3 classes
colors = ['#344AD8', '#FFFFFF', '#48804F']
cm = LinearSegmentedColormap.from_list('new', colors, N=50)

# plot the scene and the mask
fig = pyplot.figure(figsize=(20,20))
ax1 = fig.add_subplot(121)
rasterio_show(normRGB, transform=raster.transform, title='RGB visualisation', ax=ax1);
ax2 = fig.add_subplot(122)
map = ax2.imshow(y_num, cmap=cm, vmin=1) # trick to be able to plot the colorbar while using rasterio
rasterio_show(y_num, transform=raster.transform, title='Landcover classification', ax=ax2, cmap=cm, vmin=0);
divider = make_axes_locatable(ax2)
cax = divider.append_axes("right", size="2%", pad=0.05)
cbar = fig.colorbar(map, cax=cax, ticks=[1, 2, 3])
cbar.ax.set_yticklabels(['Water', 'Bare Soils', 'Vegetation']);
fig.tight_layout()

## <a name="Subsection 7.5"><font size="5" color="#1F497D">**5.5 Predictions for a stack of images**</font></a>

In [ ]:
# input the path of the folder
dataFolder = 'data/landsat_scenes'
# create a list with the paths to all the files in the input folder
paths_scenes = [os.path.join(dataFolder, item) for item in os.listdir(dataFolder) if '.tif' in item]
# sort the list
sort = np.argsort(paths_scenes)
paths_scenes = [paths_scenes[i] for i in sort]

# get the lenght of the list of cloudless scenes
n_scenes = len(paths_scenes)
# create an empty array
predictions = np.zeros((n_scenes, rows, cols))

n = 0
for path in paths_scenes:
    print_progress_bar(n+1, len(paths_scenes), str(n+1))
    # open the scene using rasterio
    raster = rasterio.open(path, 'r')
    # get the array
    scene = raster.read()
    # reshape the array
    spectra = scene.reshape(bands, cols*rows).T

    # compute and stack spectral angles values
    sam_list = []
    for item in ['Soil', 'Water']:
        ref_spectra = np.nanmedian(reference_spectra[item], axis = 0)
        sam = np.arccos(np.divide(np.dot(spectra, ref_spectra), (np.linalg.norm(spectra, axis=1) * np.linalg.norm(ref_spectra))))
        sam = sam.reshape((sam.shape[0],1))
        sam_list.append(sam)
    for sam in sam_list:
        spectra = np.hstack((spectra, sam))

    # compute NDVI (be careful of the indices in the np array, index of band 4 is 3 and index of band 3 is 2!)
    NDVI = (spectra[:,3]-spectra[:,2])/(spectra[:,3]+spectra[:,2])
    NDVI = NDVI.reshape((NDVI.shape[0],1))
    # stack NDVI values with the training samples
    spectra = np.hstack((spectra, NDVI))

    # make prediction
    y_pred = clf.predict(spectra)
    # Iterate through the prediction list to 'translate' and convert the list to a numpy array
    y_num = np.array([labels_as_numbers[i] for i in y_pred])
    # Reshape the list in order to get a 2D array
    y_num = y_num.reshape(rows, cols)
    # store the array
    predictions[n,...] = y_num
    n += 1

#del raster
del scene
del spectra
del y_num

#<a name="Section 7"><hr color="#3B8A4C"><font size="6" color="#3B8A4C">**6. Analysing changes related to mining**</font><hr color="#3B8A4C"></a>

The main noticeable effect of mining in the images is the replacement of vegetation by bare soils/rocks. Thus we can look at pixels where vegetation is replaced by bare soils.

We start by creating an empty copy of our prediction array:

In [ ]:
change = predictions*0

Now we will look at pair of images and tag vegetation pixels that where replaced by bare soils

In [ ]:
# we iterate through the scenes. we start with the second scene and compare it to previous one, and so on...
for i in range(1, predictions.shape[0]):
    before = predictions[i-1,...]
    after = predictions[i,...]
    #we look for pixels that were vegetation (id=3) before AND soil after (id=2)
    indy, indx = np.where(np.logical_and(before==3, after==2)==True)
    # we tag these pixels with ones in the change map
    change[i, indy, indx] = 1

Before ploting our changes we create a list of years from the list of paths

In [ ]:
#create a list of dates from the names of the files
dates = []
for path in paths_scenes:
    filename = os.path.basename(path)
    date = int(filename.split('_')[2])+1
    dates.append(date)
print(dates)

We can have a look at the masks:

In [ ]:
nrow = 6
ncol = 2
fig, axs = pyplot.subplots(nrow, ncol, figsize=(14,30))
for i, ax in enumerate(fig.axes):
    date1 = dates[i]
    date2 = dates[i+1]
    title = "changes between "+str(date1)+" and "+str(date2)
    rasterio_show(change[i+1, ...], title=title, ax=ax, cmap="binary", transform=raster.transform);
fig.tight_layout()

In the plots above we can see the larges patches corresponding to the activity of the mines but also a lot of noise.

Thus, our next step is to filter the noise. We will labels the regions of our masks and remove the ones with a pixel count below a defined threshold. Several trials may be needed.

In [ ]:
# Set the threshold for the minimum pixel count
threshold = 50

# we copy our change array in case our threshold is wrong:
filtered_change = copy.deepcopy(change)

# For each change mask
for i in range(change.shape[0]):
    mask = change[i, ...]
    # we create labels for each group of masked pixels (i.e. pixels equals to 1) using skimage.measure's label function
    label_image = label(mask, connectivity=2)
    # we get statistics for each label (label value, indices, number of pixels)
    val, inv, cts = np.unique(label_image.flatten(), return_inverse=True, return_counts=True)
    # we filter the labels according to a threshold (below the threshold we set the label value to 0, same as background)
    val[cts<threshold] = 0
    # recreate the label image (now filtered)
    label_image = val[inv].reshape(label_image.shape).astype('float')
    # now we simply recreate a mask and put it in our filtered_change array
    filtered_change[i, ...] = (label_image>0).astype('int')

Let's check if the threshold is adapted:

In [ ]:
fig = pyplot.figure(figsize=(20,20))
ax1 = fig.add_subplot(121)
rasterio_show(change[1, ...], transform=raster.transform, title='Before filtering', ax=ax1, cmap="binary");
ax2 = fig.add_subplot(122)
map = ax2.imshow(label_image, cmap=cm) # trick to be able to plot the colorbar while using rasterio
rasterio_show(filtered_change[1, ...], transform=raster.transform, title='After filtering', ax=ax2, cmap="binary");
fig.tight_layout();

We can now have a look at all the filtered scenes:

In [ ]:
nrow = 6
ncol = 2
fig, axs = pyplot.subplots(nrow, ncol, figsize=(14,30))
for i, ax in enumerate(fig.axes):
    date1 = dates[i]
    date2 = dates[i+1]
    title = "changes between "+str(date1)+" and "+str(date2)
    rasterio_show(filtered_change[i+1, ...], title=title, ax=ax, cmap="binary", transform=raster.transform);
fig.tight_layout()

We can combine (sum up) all the masks in order to see the total expansion of mining activities:

In [ ]:
sum_change = np.sum(filtered_change, axis=0)

fig = pyplot.figure(figsize=(20,20))
ax1 = fig.add_subplot(121)
rasterio_show(sum_change, transform=raster.transform, title='Extent of mountaintop mining between 1986 and 2022', ax=ax1, cmap="binary", vmin=0, vmax=1);

We can also create a "fancier" version by coloring the latest year of activity:

In [ ]:
change_by_year = np.zeros((change.shape[1], change.shape[2]))

for i in range(len(dates)):
    y,x = np.where(filtered_change[i,...]==1)
    change_by_year[y, x] = i

colors = ['#f80c12','#ee1100','#ff3311','#ff4422','#ff6644','#ff9933',
          '#feae2d','#ccbb33','#d0c310','#aacc22','#69d025','#22ccaa','#12bdb9',
          '#11aabb','#4444dd','#3311bb','#3b0cbd','#442299','#ffffff']
cm = LinearSegmentedColormap.from_list('new', colors[::-1], N=100)

fig = pyplot.figure(figsize=(10,10))
ax1 = fig.add_subplot(111)
map = pyplot.imshow(change_by_year, cmap = cm, interpolation=None)
rasterio_show(change_by_year, transform=raster.transform, title='Mountaintop mining - year of deforestation', ax=ax1, cmap=cm, vmin=0, interpolation=None);
divider = make_axes_locatable(ax1)
cax = divider.append_axes("right", size="2%", pad=0.05)
cbar = fig.colorbar(map, cax=cax, ticks=[i for i in range(len(dates))])
cbar.ax.set_yticklabels([str(i) for i in dates]);
fig.tight_layout()

To conclude, we can calculate how much areas were deforested each year and plot the result:

In [ ]:
mined_areas = []

# compute the total area per year
# pixel resolution is 30m so area = 900 m2 = 0.0009 km2
for i in range(len(dates)):
    area_meters = np.sum(filtered_change[i,...])*0.0009
    mined_areas.append(area_meters)

# plot the figure
fig = pyplot.figure(figsize=(24, 6))
ax = fig.add_subplot(1, 2, 1)
ax.plot(dates[1::], mined_areas[1::], 'r-o', label = "Deforested areas")
ax.set_xlim([1985,2025])
ax.set_ylim([0,70])
ax.set_ylabel("Deforested areas (km2)")
pyplot.title("Deforestation (km2) resulting from mountaintop mining since 1985", fontsize=12)
pyplot.legend();

ax = fig.add_subplot(1, 2, 2)
ax.plot(dates, np.cumsum(mined_areas), 'r-o', label = "Cumulative curve")
ax.set_xlim([1985,2025])
ax.set_ylim([0,450])
ax.set_ylabel("Deforested areas (km2)")
pyplot.title("Cumulated deforestation (km2) resulting from mountaintop mining since 1985", fontsize=12)
pyplot.legend();